# NBA Game-Prediction Dataset Builder (NBA API, last 10 seasons)

Replaces the SQLite-from-Kaggle source with **live data from `nba_api`**, so it stays current.

**Two phases:**

1. **Phase 1 — Box scores.** Fetch team-level game logs from the NBA stats API for each season, stack them, and produce `box_data.csv` in the same T1/T2 layout (T1=home, T2=away).
2. **Phase 2 — Pre-game features.** Identical leakage-free pipeline as the previous notebook: each game's features use only that team's prior games in the same season.

**Outputs:**
- `box_data.csv` — game-level T1/T2 box scores
- `matchups.csv` — pre-game season-to-date features for both teams + `Target_Win`

**Phase 1 caches each season's raw fetch to `raw_seasons/`** so you don't pay the network cost twice if you re-run.


## Setup

In [1]:
# One-time install if you don't have it:
# !pip install nba_api pandas numpy

import time
from pathlib import Path

import numpy as np
import pandas as pd
from nba_api.stats.endpoints import leaguegamelog

# ---- Config ----
# 10 seasons (last full decade). Edit if you want a different range.
SEASONS = [f"{y}-{str(y+1)[2:]}" for y in range(2016, 2026)]   # 2016-17 .. 2025-26
SEASON_TYPE = "Regular Season"   # or "Playoffs"
RAW_DIR     = Path("raw_seasons")
OUT_DIR     = Path(".")
RAW_DIR.mkdir(exist_ok=True)

SLEEP_BETWEEN_CALLS = 0.6        # be polite to stats.nba.com
RETRIES             = 4

# Same Elo constants as before
ELO_BASE      = 1500.0
ELO_K         = 20.0
ELO_HOME_ADV  = 100.0
ELO_REGRESS   = 0.25

STAT_COLS = ['Score','FGM','FGA','FGM3','FGA3','FTM','FTA',
             'OR','DR','Ast','TO','Stl','Blk','PF']
OPP_COLS  = [f'Opp_{s}' for s in STAT_COLS]

print("Seasons to fetch:", SEASONS)

Seasons to fetch: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']


## Phase 1 — Fetch & build `box_data.csv`

`LeagueGameLog` returns one row per (team, game), so 2 rows per game. We pivot home vs. away (the `MATCHUP` field uses `vs.` for home and `@` for away) and merge into one row per game with T1_*/T2_* columns.

### Helper: fetch one season with retries + cache

In [2]:
def fetch_season(season: str,
                 season_type: str = SEASON_TYPE,
                 retries: int = RETRIES,
                 cache_dir: Path = RAW_DIR) -> pd.DataFrame:
    """Pull team game logs for a single season. Caches to a CSV in cache_dir."""
    fname = cache_dir / f"{season}_{season_type.replace(' ', '_')}.csv"
    if fname.exists():
        return pd.read_csv(fname)

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            log = leaguegamelog.LeagueGameLog(
                season=season,
                season_type_all_star=season_type,
                player_or_team_abbreviation='T',  # team-level
                timeout=60,
            )
            df = log.get_data_frames()[0]
            df.to_csv(fname, index=False)
            return df
        except Exception as e:
            last_err = e
            wait = 5 * attempt
            print(f"  [{season}] attempt {attempt}/{retries} failed: {e}; "
                  f"retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch {season} after {retries} attempts: {last_err}")

### Run the fetch loop

In [3]:
raw_frames = []
for s in SEASONS:
    print(f"Fetching {s} ...", end=" ", flush=True)
    df = fetch_season(s)
    print(f"{len(df):,} team-game rows")
    raw_frames.append(df)
    time.sleep(SLEEP_BETWEEN_CALLS)

raw = pd.concat(raw_frames, ignore_index=True)
print(f"\nTotal team-game rows: {len(raw):,}")
raw.head()

Fetching 2016-17 ... 2,460 team-game rows
Fetching 2017-18 ... 2,460 team-game rows
Fetching 2018-19 ... 2,460 team-game rows
Fetching 2019-20 ... 2,118 team-game rows
Fetching 2020-21 ... 2,160 team-game rows
Fetching 2021-22 ... 2,460 team-game rows
Fetching 2022-23 ... 2,460 team-game rows
Fetching 2023-24 ... 2,460 team-game rows
Fetching 2024-25 ... 2,460 team-game rows
Fetching 2025-26 ... 2,460 team-game rows

Total team-game rows: 23,958


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22016,1610612752,NYK,New York Knicks,21600001,2016-10-25,NYK @ CLE,L,240,32,...,29,42,17,6,6,18,22,88,-29,1
1,22016,1610612757,POR,Portland Trail Blazers,21600002,2016-10-25,POR vs. UTA,W,240,39,...,29,34,22,5,3,13,18,113,9,1
2,22016,1610612759,SAS,San Antonio Spurs,21600003,2016-10-25,SAS @ GSW,W,240,47,...,34,55,25,13,3,14,19,129,29,1
3,22016,1610612739,CLE,Cleveland Cavaliers,21600001,2016-10-25,CLE vs. NYK,W,240,45,...,40,51,31,12,5,15,22,117,29,1
4,22016,1610612744,GSW,Golden State Warriors,21600003,2016-10-25,GSW vs. SAS,L,240,40,...,27,35,24,11,6,16,19,100,-29,1


### Helper: stack team-game rows into the T1/T2 box layout

`MATCHUP` like `"LAL vs. BOS"` = home (T1), `"LAL @ BOS"` = away (T2). We split, rename columns, and inner-join on `GAME_ID`.

In [4]:
# Map of nba_api column names -> our schema (without the T1_/T2_ prefix)
NBA_TO_OURS = {
    'TEAM_ID':   'TeamID',
    'TEAM_NAME': 'TeamName',
    'PTS':  'Score',
    'FGM':  'FGM',  'FGA':  'FGA',
    'FG3M': 'FGM3', 'FG3A': 'FGA3',
    'FTM':  'FTM',  'FTA':  'FTA',
    'OREB': 'OR',   'DREB': 'DR',
    'AST':  'Ast',  'TOV':  'TO',
    'STL':  'Stl',  'BLK':  'Blk',
    'PF':   'PF',
}


def build_box_data(raw: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Convert nba_api LeagueGameLog rows into the T1/T2 box layout."""
    df = raw.copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    # SEASON_ID like '22015' -> Season = 2015 (start year of the 2015-16 season)
    df['Season'] = df['SEASON_ID'].astype(str).str[-4:].astype(int)

    # Keep only standard regular-season matchup strings ("X vs. Y" or "X @ Y").
    # All-Star, preseason oddities, etc. get dropped here.
    mask = df['MATCHUP'].str.contains(r' vs\. | @ ', regex=True, na=False)
    n_dropped = (~mask).sum()
    if n_dropped and verbose:
        print(f"  dropped {n_dropped} rows with non-standard MATCHUP")
    df = df[mask].copy()
    df['IS_HOME'] = df['MATCHUP'].str.contains(' vs. ', regex=False)

    # First-pass dedupe: same (game, team) appearing twice in the API response
    before = len(df)
    df = df.drop_duplicates(subset=['GAME_ID', 'TEAM_ID'])
    if verbose and len(df) < before:
        print(f"  dropped {before - len(df)} duplicate (GAME_ID, TEAM_ID) rows")

    # Drop rows missing core box-score numbers
    needed = list(NBA_TO_OURS.keys()) + ['WL']
    df = df.dropna(subset=needed)

    # Diagnose any GAME_IDs that don't have exactly 1 home + 1 away row.
    # This catches rare data glitches that would otherwise blow up the merge.
    counts = df.groupby('GAME_ID')['IS_HOME'].agg(home_rows='sum', total='count')
    bad = counts[(counts['home_rows'] != 1) | (counts['total'] != 2)]
    if len(bad) > 0:
        if verbose:
            print(f"  dropping {len(bad)} games with unusual home/away structure")
            print(f"    example GAME_IDs: {list(bad.index[:5])}")
        df = df[~df['GAME_ID'].isin(bad.index)]

    cols_to_keep = ['GAME_ID', 'GAME_DATE', 'Season', 'WL'] + list(NBA_TO_OURS.keys())

    home = (df[df['IS_HOME']][cols_to_keep]
            .rename(columns={**{k: f'T1_{v}' for k, v in NBA_TO_OURS.items()},
                             'WL': 'T1_WL'}))
    away = (df[~df['IS_HOME']][['GAME_ID', 'WL'] + list(NBA_TO_OURS.keys())]
            .rename(columns={**{k: f'T2_{v}' for k, v in NBA_TO_OURS.items()},
                             'WL': 'T2_WL'}))

    # Clean enough now that an inner merge gives exactly the games we want.
    box = home.merge(away, on='GAME_ID', how='inner')

    box = box.rename(columns={'GAME_ID': 'GameID', 'GAME_DATE': 'DayDate'})
    box['T1_Loc']     = 'H'
    box['Target_Win'] = (box['T1_WL'] == 'W').astype(int)
    box = box.drop(columns=['T1_WL', 'T2_WL'])

    # Order columns the same way the SQLite version did
    front = ['Season', 'DayDate', 'GameID',
             'T1_TeamID', 'T1_TeamName', 'T2_TeamID', 'T2_TeamName', 'T1_Loc']
    t1_stats = [f'T1_{c}' for c in STAT_COLS]
    t2_stats = [f'T2_{c}' for c in STAT_COLS]
    box = box[front + t1_stats + t2_stats + ['Target_Win']]

    box = box.sort_values(['DayDate', 'GameID']).reset_index(drop=True)
    return box

In [5]:
box = build_box_data(raw)
box.to_csv(OUT_DIR / "box_data.csv", index=False)
print(f"Wrote box_data.csv ({len(box):,} games, "
      f"seasons {box['Season'].min()}-{box['Season'].max()})")
box.head()

  dropping 10 games with unusual home/away structure
    example GAME_IDs: [22400147, 22400621, 22400633, 22401229, 22401230]
Wrote box_data.csv (11,969 games, seasons 2016-2025)


,Season,DayDate,GameID,T1_TeamID,T1_TeamName,T2_TeamID,T2_TeamName,T1_Loc,T1_Score,T1_FGM,...,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,Target_Win
0,2016,2016-10-25,21600001,1610612739,Cleveland Cavaliers,1610612752,New York Knicks,H,117,45,...,15,20,13,29,17,18,6,6,22,1
1,2016,2016-10-25,21600002,1610612757,Portland Trail Blazers,1610612762,Utah Jazz,H,113,39,...,16,16,6,25,19,14,9,5,19,1
2,2016,2016-10-25,21600003,1610612744,Golden State Warriors,1610612759,San Antonio Spurs,H,100,40,...,23,26,21,34,25,14,13,3,19,0
3,2016,2016-10-26,21600004,1610612753,Orlando Magic,1610612748,Miami Heat,H,96,34,...,10,16,16,36,27,12,5,7,22,0
4,2016,2016-10-26,21600005,1610612754,Indiana Pacers,1610612742,Dallas Mavericks,H,130,47,...,13,18,10,39,26,15,8,8,27,1


## Phase 2 — Pre-game features

Identical leakage-free pipeline from before. For every `(GameID, TeamID)`, compute season-to-date averages, win pcts, advanced metrics, last-14-day form, pre-game Elo, and a snapshot seed — all using **only games strictly before** the current game in the same season.

The first game of each team-season has no priors, so its features come out as `NaN`.

### Helpers

In [6]:
def explode_to_team_perspective(box: pd.DataFrame) -> pd.DataFrame:
    """Each game becomes 2 rows -- one from each team's perspective."""
    def side(prefix_self, prefix_opp, loc):
        d = pd.DataFrame({
            'Season' : box['Season'],
            'DayDate': box['DayDate'],
            'GameID' : box['GameID'],
            'TeamID' : box[f'{prefix_self}_TeamID'],
            'OppID'  : box[f'{prefix_opp}_TeamID'],
            'Loc'    : loc,
            'Won'    : box['Target_Win'] if prefix_self == 'T1' else 1 - box['Target_Win'],
        })
        for s in STAT_COLS:
            d[s]          = box[f'{prefix_self}_{s}']
            d[f'Opp_{s}'] = box[f'{prefix_opp}_{s}']
        return d

    return pd.concat([side('T1', 'T2', 'H'),
                      side('T2', 'T1', 'A')], ignore_index=True)


def compute_possessions(g: pd.DataFrame) -> pd.DataFrame:
    """Per-game possessions (Poss = FGA - OR + TO + 0.475*FTA)."""
    g = g.copy()
    g['Poss']    = g['FGA']     - g['OR']     + g['TO']     + 0.475 * g['FTA']
    g['OppPoss'] = g['Opp_FGA'] - g['Opp_OR'] + g['Opp_TO'] + 0.475 * g['Opp_FTA']
    return g

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

def calculate_advanced_elo_and_sos(base_df):
    """
    Chronologically calculates Margin-of-Victory adjusted Elo and 
    Strength of Schedule (average opponent Pregame Elo).
    """
    # 1. Isolate unique games so we only process each matchup once
    games = base_df.drop_duplicates(subset=['GameID']).sort_values(['Season', 'DayDate', 'GameID']).copy()
    
    # Trackers
    elo_dict = defaultdict(lambda: 1500.0) # Everyone starts at 1500
    sos_dict = defaultdict(list)           # Tracks past opponent Elos for the current season
    last_season = defaultdict(int)         # Tracks season changes for mean reversion
    
    results = []
    
    K = 20.0
    HOME_ADV = 100.0
    
    for _, row in games.iterrows():
        game_id = row['GameID']
        season = row['Season']
        team_a = row['TeamID']
        team_b = row['OppID']
        
        # --- Season Mean Reversion ---
        # Pull teams 25% back towards average (1500) at the start of a new season
        for t in [team_a, team_b]:
            if last_season[t] != season and last_season[t] != 0:
                elo_dict[t] = (0.75 * elo_dict[t]) + (0.25 * 1500.0)
                sos_dict[(t, season)] = [] # Reset SoS history for the new season
            last_season[t] = season
            
        # --- 1. Get Pregame Elos ---
        elo_a = elo_dict[team_a]
        elo_b = elo_dict[team_b]
        
        # --- 2. Get Pregame SoS (Strength of Schedule) ---
        # Average of all opponent Elos faced THIS season prior to this game
        sos_a = np.mean(sos_dict[(team_a, season)]) if sos_dict[(team_a, season)] else 1500.0
        sos_b = np.mean(sos_dict[(team_b, season)]) if sos_dict[(team_b, season)] else 1500.0
        
        # Store the pregame stats for merging later
        results.append({'GameID': game_id, 'TeamID': team_a, 'Pregame_Elo': elo_a, 'Pregame_SoS': sos_a})
        results.append({'GameID': game_id, 'TeamID': team_b, 'Pregame_Elo': elo_b, 'Pregame_SoS': sos_b})
        
        # --- Update Trackers ---
        sos_dict[(team_a, season)].append(elo_b)
        sos_dict[(team_b, season)].append(elo_a)
        
        # --- 3. ELO UPDATE MATH ---
        # Calculate Expected Win Probabilities
        loc_a = row['Loc']
        elo_a_adj = elo_a + HOME_ADV if loc_a == 'H' else elo_a
        elo_b_adj = elo_b + HOME_ADV if loc_a == 'A' else elo_b
        
        exp_a = 1 / (1 + 10 ** ((elo_b_adj - elo_a_adj) / 400.0))
        exp_b = 1 - exp_a
        
        # Actual outcome
        score_a = row['Score']
        score_b = row['Opp_Score']
        margin = abs(score_a - score_b)
        win_a = 1 if score_a > score_b else 0
        win_b = 1 - win_a
        
        # --- THE 538 AUTOCORRELATION FIX ---
        # 1. Figure out the Elo difference from the WINNER'S perspective
        elo_winner = elo_a_adj if win_a == 1 else elo_b_adj
        elo_loser  = elo_b_adj if win_a == 1 else elo_a_adj
        elo_diff   = elo_winner - elo_loser 
        
        # 2. Apply FiveThirtyEight's official MoV formula
        # If an underdog wins, elo_diff is negative -> denominator gets smaller -> multiplier gets HUGE!
        numerator = (margin + 3.0) ** 0.8
        denominator = 7.5 + 0.006 * elo_diff
        mov_multiplier = numerator / denominator
        
        # Apply the MoV multiplier to the Elo shift
        elo_dict[team_a] += K * mov_multiplier * (win_a - exp_a)
        elo_dict[team_b] += K * mov_multiplier * (win_b - exp_b)

    return pd.DataFrame(results)

### Main computation

In [14]:
def compute_pregame_features(box: pd.DataFrame) -> pd.DataFrame:
    tg = explode_to_team_perspective(box)
    tg = compute_possessions(tg)
    tg = tg.sort_values(['TeamID', 'Season', 'DayDate', 'GameID']).reset_index(drop=True)

    grp_keys = ['TeamID', 'Season']
    grp = tg.groupby(grp_keys)

    # ---- Games played BEFORE this one (0 for first game of season) ----
    tg['Games_Played'] = grp.cumcount()

    # ---- Cumulative totals BEFORE this game (cumsum - current) ----
    sum_cols = STAT_COLS + OPP_COLS + ['Poss', 'OppPoss']
    for c in sum_cols:
        tg[f'cum_{c}'] = grp[c].cumsum() - tg[c]
    tg['cum_Wins'] = grp['Won'].cumsum() - tg['Won']

    # ---- Per-game averages ----
    safe_games = tg['Games_Played'].replace(0, np.nan)
    for c in STAT_COLS + OPP_COLS:
        tg[f'Avg_{c}'] = tg[f'cum_{c}'] / safe_games

    # ---- Advanced metrics on cumulative totals ----
    safe_poss     = tg['cum_Poss'].replace(0, np.nan)
    safe_opp_poss = tg['cum_OppPoss'].replace(0, np.nan)
    safe_fga      = tg['cum_FGA'].replace(0, np.nan)
    safe_or_dr    = (tg['cum_OR'] + tg['cum_Opp_DR']).replace(0, np.nan)

    tg['Avg_Off_Eff'] = (tg['cum_Score']     / safe_poss)     * 100   # OffEff
    tg['Avg_Def_Eff'] = (tg['cum_Opp_Score'] / safe_opp_poss) * 100   # DefEff
    tg['Net_Rating']  = tg['Avg_Off_Eff'] - tg['Avg_Def_Eff']         # NetEff
    tg['EFG_Pct']     = (tg['cum_FGM'] + 0.5 * tg['cum_FGM3']) / safe_fga
    tg['TO_Rate']     = tg['cum_TO'] / safe_poss
    tg['OR_Pct']      = tg['cum_OR'] / safe_or_dr

    # ---- Win pct (overall / home / away) BEFORE current game ----
    tg['Win_Pct']         = tg['cum_Wins'] / safe_games
    tg['Overall_Win_Pct'] = tg['Win_Pct']

    tg['_is_home']  = (tg['Loc'] == 'H').astype(int)
    tg['_is_away']  = (tg['Loc'] == 'A').astype(int)
    tg['_home_won'] = tg['Won'] * tg['_is_home']
    tg['_away_won'] = tg['Won'] * tg['_is_away']
    g2 = tg.groupby(grp_keys)
    cum_hg = g2['_is_home'].cumsum()  - tg['_is_home']
    cum_hw = g2['_home_won'].cumsum() - tg['_home_won']
    cum_ag = g2['_is_away'].cumsum()  - tg['_is_away']
    cum_aw = g2['_away_won'].cumsum() - tg['_away_won']
    tg['Home_Win_Pct'] = cum_hw / cum_hg.replace(0, np.nan)
    tg['Away_Win_Pct'] = cum_aw / cum_ag.replace(0, np.nan)
    tg = tg.drop(columns=['_is_home', '_is_away', '_home_won', '_away_won'])

    # ---- Last-14-day win pct (rolling time window, current row excluded) ----
    last14 = np.full(len(tg), np.nan)
    for _, idx in tg.groupby(grp_keys, sort=False).indices.items():
        sub = tg.iloc[idx]
        s = pd.Series(sub['Won'].values,
                      index=pd.DatetimeIndex(sub['DayDate']))
        last14[idx] = s.rolling('14D', closed='left').mean().values
    tg['Last_14_Days_Win_Pct'] = last14

    # ---- Pre-game Elo & SoS (merged from chronological walk) ----
    # Pass 'tg' (which already has Loc, Score, Opp_Score) into our new function
    elo_df = calculate_advanced_elo_and_sos(tg)
    
    # It already outputs GameID, TeamID, Pregame_Elo, and Pregame_SoS!
    # So we just merge it straight in:
    tg = tg.merge(elo_df, on=['GameID', 'TeamID'], how='left')

    # ---- Seed: rank by current Win_Pct within season as of game date ----
    seed_parts = []
    for season, sub in tg.groupby('Season'):
        pvt = (sub.pivot_table(index='DayDate', columns='TeamID',
                               values='Win_Pct', aggfunc='mean')
                  .sort_index().ffill())
        ranks = pvt.rank(axis=1, method='min', ascending=False)
        long = (ranks.reset_index()
                     .melt(id_vars='DayDate', var_name='TeamID', value_name='seed'))
        long['Season'] = season
        seed_parts.append(long)
    seeds = pd.concat(seed_parts, ignore_index=True)
    tg = tg.merge(seeds, on=['Season','TeamID','DayDate'], how='left')

    keep = (['Season','DayDate','GameID','TeamID','OppID','Loc','Won','Games_Played']
            + [f'Avg_{c}' for c in STAT_COLS + OPP_COLS]
            + ['Avg_Off_Eff','Avg_Def_Eff','Net_Rating','EFG_Pct','TO_Rate','OR_Pct',
               'Win_Pct','Overall_Win_Pct','Home_Win_Pct','Away_Win_Pct',
               'Last_14_Days_Win_Pct','Pregame_Elo','seed'])
    return tg[keep].reset_index(drop=True)

In [15]:
pregame = compute_pregame_features(box)
print(f"Pre-game features: {len(pregame):,} rows ({len(pregame)//2:,} games x 2 teams)")
print(f"Rows with NaN features (first game of each team-season): "
      f"{pregame['Avg_Score'].isna().sum():,}")
pregame.head()

Pre-game features: 23,938 rows (11,969 games x 2 teams)
Rows with NaN features (first game of each team-season): 300


,Season,DayDate,GameID,TeamID,OppID,Loc,Won,Games_Played,Avg_Score,Avg_FGM,...,EFG_Pct,TO_Rate,OR_Pct,Win_Pct,Overall_Win_Pct,Home_Win_Pct,Away_Win_Pct,Last_14_Days_Win_Pct,Pregame_Elo,seed
0,2016,2016-10-27,21600014,1610612737,1610612764,H,1,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.000000,NaN
1,2016,2016-10-29,21600026,1610612737,1610612755,A,1,1,114.0,44.000000,...,0.568182,0.202801,0.333333,1.00,1.00,1.000000,NaN,1.00,1508.974021,1.0
2,2016,2016-10-31,21600044,1610612737,1610612758,H,1,2,109.0,43.000000,...,0.546243,0.160603,0.240964,1.00,1.00,1.000000,1.0,1.00,1539.025165,1.0
3,2016,2016-11-02,21600059,1610612737,1610612747,H,0,3,108.0,39.666667,...,0.527778,0.151915,0.280303,1.00,1.00,1.000000,1.0,1.00,1545.555387,1.0
4,2016,2016-11-04,21600070,1610612737,1610612764,A,0,4,110.0,40.000000,...,0.532641,0.156260,0.271676,0.75,0.75,0.666667,1.0,0.75,1531.717314,5.0


## Phase 3 — Build `matchups.csv`

Merge each game with both teams' pre-game features, in the column order from your spec (with `End_Season_Elo` -> `Pregame_Elo`, since the Elo is now snapshotted before each game rather than at season-end).

In [16]:
FEATURE_COLS = [
    'Avg_Off_Eff', 'Avg_Def_Eff', 'Net_Rating',
    'Win_Pct', 'Pregame_Elo',
    'Overall_Win_Pct', 'Home_Win_Pct', 'Away_Win_Pct',
    'Avg_Score', 'Avg_FGM', 'Avg_FGA', 'Avg_FGM3', 'Avg_FGA3',
    'Avg_FTM',   'Avg_FTA', 'Avg_OR',  'Avg_DR',
    'Avg_Ast',   'Avg_TO',  'Avg_Stl', 'Avg_Blk', 'Avg_PF',
    'Avg_Opp_Score', 'Avg_Opp_FGM', 'Avg_Opp_FGA',
    'Avg_Opp_FGM3',  'Avg_Opp_FGA3',
    'Avg_Opp_FTM',   'Avg_Opp_FTA',
    'Avg_Opp_OR',    'Avg_Opp_DR',   'Avg_Opp_Ast',
    'Avg_Opp_TO',    'Avg_Opp_Stl',  'Avg_Opp_Blk', 'Avg_Opp_PF',
    'seed', 'Last_14_Days_Win_Pct',
]


def build_matchups(box: pd.DataFrame, pregame: pd.DataFrame) -> pd.DataFrame:
    feats = pregame[['GameID', 'TeamID'] + FEATURE_COLS]

    t1 = feats.rename(columns={'TeamID': 'T1_TeamID',
                               **{c: f'T1_{c}' for c in FEATURE_COLS}})
    t2 = feats.rename(columns={'TeamID': 'T2_TeamID',
                               **{c: f'T2_{c}' for c in FEATURE_COLS}})

    m = (box[['Season','DayDate','GameID','T1_TeamID','T2_TeamID','Target_Win']]
            .merge(t1, on=['GameID', 'T1_TeamID'])
            .merge(t2, on=['GameID', 'T2_TeamID']))

    m['ID'] = (m['Season'].astype(str) + '_'
               + m['T1_TeamID'].astype(str) + '_'
               + m['T2_TeamID'].astype(str))

    ordered = (['T1_TeamID', 'T2_TeamID', 'Season', 'DayDate', 'GameID', 'ID']
               + [f'T1_{c}' for c in FEATURE_COLS]
               + [f'T2_{c}' for c in FEATURE_COLS]
               + ['Target_Win'])
    return m[ordered]

In [17]:
matchups = build_matchups(box, pregame)
matchups.to_csv(OUT_DIR / "matchups.csv", index=False)
print(f"Wrote matchups.csv ({len(matchups):,} rows, {len(matchups.columns)} cols)")

# Optional: drop the first-game-of-season rows that have NaN features
clean = matchups.dropna(subset=[c for c in matchups.columns
                                if c.startswith(('T1_Avg', 'T2_Avg'))])
print(f"Rows after dropping NaN-feature games: {len(clean):,}")
matchups.head()

Wrote matchups.csv (11,969 rows, 83 cols)
Rows after dropping NaN-feature games: 11,810


,T1_TeamID,T2_TeamID,Season,DayDate,GameID,ID,T1_Avg_Off_Eff,T1_Avg_Def_Eff,T1_Net_Rating,T1_Win_Pct,...,T2_Avg_Opp_OR,T2_Avg_Opp_DR,T2_Avg_Opp_Ast,T2_Avg_Opp_TO,T2_Avg_Opp_Stl,T2_Avg_Opp_Blk,T2_Avg_Opp_PF,T2_seed,T2_Last_14_Days_Win_Pct,Target_Win
0,1610612739,1610612752,2016,2016-10-25,21600001,2016_1610612739_1610612752,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,1610612757,1610612762,2016,2016-10-25,21600002,2016_1610612757_1610612762,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,1610612744,1610612759,2016,2016-10-25,21600003,2016_1610612744_1610612759,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,1610612753,1610612748,2016,2016-10-26,21600004,2016_1610612753_1610612748,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1610612754,1610612742,2016,2016-10-26,21600005,2016_1610612754_1610612742,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


## Sanity checks

Same checks as before — `Games_Played` should start at 0 and increment by 1, and `Avg_Score` for any given pre-game row should equal the manual mean of that team's prior games' scores.

In [18]:
gp_check = (pregame.sort_values(['TeamID','Season','DayDate','GameID'])
                   .groupby(['TeamID','Season'])['Games_Played']
                   .agg(['min','max','count']))
print("Games_Played min should be 0 for every team-season:",
      (gp_check['min'] == 0).all())
print("Games_Played max should equal count - 1:",
      (gp_check['max'] == gp_check['count'] - 1).all())

team_log = (explode_to_team_perspective(box)
            .sort_values(['TeamID','Season','DayDate','GameID']))
sample = (pregame[pregame['Games_Played'] >= 4]
          .sort_values(['TeamID','Season','DayDate','GameID'])
          .head(1).iloc[0])

prior = team_log[(team_log['TeamID'] == sample['TeamID']) &
                 (team_log['Season'] == sample['Season']) &
                 (team_log['DayDate'] < sample['DayDate'])]
print(f"\nSpot check for TeamID={sample['TeamID']}, Season={sample['Season']}, "
      f"GameID={sample['GameID']}")
print(f"  Pre-game Avg_Score in pregame:        {sample['Avg_Score']:.3f}")
print(f"  Manual mean of prior games' Score:    {prior['Score'].mean():.3f}")

Games_Played min should be 0 for every team-season: True
Games_Played max should equal count - 1: True

Spot check for TeamID=1610612737, Season=2016, GameID=21600070
  Pre-game Avg_Score in pregame:        110.000
  Manual mean of prior games' Score:    110.000


## Done

You now have:

- `box_data.csv` — game-level T1/T2 box scores from the last ~10 NBA seasons
- `matchups.csv` — pre-game features for both teams, ready for modeling

`raw_seasons/` holds per-season caches in case you want to refresh just the current season later.

**To refresh the current season**: delete `raw_seasons/2025-26_Regular_Season.csv` and re-run the fetch loop. The other seasons stay cached.
